In [1]:
import torch
import numpy as np
import cv2
from facenet_pytorch import MTCNN

print("✅ PyTorch Version:", torch.__version__)
print("✅ CUDA Available:", torch.cuda.is_available())
print("✅ GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
print("✅ NumPy Version:", np.__version__)
print("✅ OpenCV Version:", cv2.__version__)
print("✅ Facenet-PyTorch Loaded:", MTCNN is not None)


c:\Users\bchal\anaconda3\envs\mindcare-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch Version: 2.6.0+cu126
✅ CUDA Available: True
✅ GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
✅ NumPy Version: 1.26.4
✅ OpenCV Version: 4.11.0
✅ Facenet-PyTorch Loaded: True


In [ ]:
from flask import Flask, request, jsonify
import cv2
import numpy as np
import torch
import base64
from facenet_pytorch import MTCNN
from scipy.signal import butter, filtfilt
import io
from PIL import Image

app = Flask(__name__)

# Initialize MTCNN for face detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(keep_all=False, device=device)

# Buffer pour stocker les valeurs
buffer_storage = {}  # Format: {session_id: {'r': [], 'g': [], 'b': [], 'bpm_history': []}}

def butter_bandpass(lowcut, highcut, fs, order=5):
    """Design a butterworth bandpass filter."""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    """Apply bandpass filter to data."""
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data)
    return y

@app.route('/analyze', methods=['POST'])
def analyze_heart_rate():
    try:
        # Récupérer les données de la requête
        data = request.json
        if not data or 'image' not in data or 'session_id' not in data:
            return jsonify({'error': 'Image data and session_id required'}), 400
        
        session_id = data['session_id']
        image_b64 = data['image']
        fps = data.get('fps', 30)  # Utiliser 30 comme valeur par défaut
        
        # Initialiser le stockage pour cette session si nécessaire
        if session_id not in buffer_storage:
            buffer_storage[session_id] = {
                'r': [], 'g': [], 'b': [], 'bpm_history': []
            }
        
        # Décodage de l'image base64
        img_data = base64.b64decode(image_b64.split(',')[1] if ',' in image_b64 else image_b64)
        nparr = np.frombuffer(img_data, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        # Convertir en RGB pour MTCNN
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Détecter le visage
        boxes, _ = mtcnn.detect(rgb_frame)
        
        response = {
            'face_detected': False,
            'bpm': None,
            'quality': None,
            'buffer_size': 0,
            'buffer_full': False
        }
        
        if boxes is not None and boxes.shape[0] > 0:
            response['face_detected'] = True
            box = boxes[0]  # Utiliser le premier visage détecté
            x1, y1, x2, y2 = map(int, box)
            
            # Extraire la ROI de la joue (meilleure pour le signal PPG)
            face_height = y2 - y1
            face_width = x2 - x1
            
            cheek_y1 = y1 + int(face_height * 0.3)
            cheek_y2 = y1 + int(face_height * 0.7)
            cheek_x1 = x1 + int(face_width * 0.2)
            cheek_x2 = x2 - int(face_width * 0.2)
            
            # S'assurer que la ROI est dans les limites du cadre
            cheek_y1 = max(0, cheek_y1)
            cheek_y2 = min(frame.shape[0], cheek_y2)
            cheek_x1 = max(0, cheek_x1)
            cheek_x2 = min(frame.shape[1], cheek_x2)
            
            # Extraire ROI
            if cheek_y2 > cheek_y1 and cheek_x2 > cheek_x1:
                roi = frame[cheek_y1:cheek_y2, cheek_x1:cheek_x2]
                
                # Extraire les valeurs RGB moyennes
                r = np.mean(roi[:, :, 2])  # Canal rouge en BGR
                g = np.mean(roi[:, :, 1])  # Canal vert en BGR
                b = np.mean(roi[:, :, 0])  # Canal bleu en BGR
                
                # Stocker dans les buffers
                buffer_storage[session_id]['r'].append(r)
                buffer_storage[session_id]['g'].append(g)
                buffer_storage[session_id]['b'].append(b)
                
                # Taille maximale du buffer = 10 secondes de données
                frame_count = int(fps * 10)
                
                # Limiter la taille du buffer
                if len(buffer_storage[session_id]['g']) > frame_count:
                    buffer_storage[session_id]['r'].pop(0)
                    buffer_storage[session_id]['g'].pop(0)
                    buffer_storage[session_id]['b'].pop(0)
                
                response['buffer_size'] = len(buffer_storage[session_id]['g'])
                response['buffer_full'] = len(buffer_storage[session_id]['g']) == frame_count
                
                # Calculer BPM si on a assez de données
                if len(buffer_storage[session_id]['g']) >= frame_count * 0.7:  # Au moins 70% rempli
                    # Utiliser le canal vert pour un meilleur signal
                    signal = np.array(buffer_storage[session_id]['g'])
                    
                    # Prétraitement du signal
                    detrended = signal - np.mean(signal)
                    normalized = detrended / np.std(detrended)
                    
                    # Filtrage passe-bande (0.7-3.5 Hz correspond à 42-210 BPM)
                    filtered = butter_bandpass_filter(normalized, 0.7, 3.5, fps)
                    
                    # Appliquer FFT
                    fft_data = np.abs(np.fft.rfft(filtered))
                    frequencies = np.fft.rfftfreq(len(filtered), 1/fps)
                    
                    # Trouver la plage de fréquence valide (0.7 - 3.5 Hz)
                    valid_idx = np.where((frequencies >= 0.7) & (frequencies <= 3.5))
                    valid_freq = frequencies[valid_idx]
                    valid_fft = fft_data[valid_idx]
                    
                    if len(valid_freq) > 0 and np.max(valid_fft) > 0:
                        # Trouver la fréquence pic
                        peak_idx = np.argmax(valid_fft)
                        peak_freq = valid_freq[peak_idx]
                        
                        # Convertir en BPM
                        bpm = peak_freq * 60
                        
                        # Ajouter à l'historique pour lissage
                        buffer_storage[session_id]['bpm_history'].append(bpm)
                        if len(buffer_storage[session_id]['bpm_history']) > 10:
                            buffer_storage[session_id]['bpm_history'].pop(0)
                        
                        # Utiliser la médiane pour la stabilité
                        if len(buffer_storage[session_id]['bpm_history']) >= 3:
                            smoothed_bpm = np.median(buffer_storage[session_id]['bpm_history'])
                            response['bpm'] = int(smoothed_bpm)
                            
                            # Indicateur de qualité du signal
                            signal_strength = np.max(valid_fft) / np.mean(valid_fft)
                            if signal_strength > 10:
                                response['quality'] = "Excellent"
                            elif signal_strength > 5:
                                response['quality'] = "Good"
                            else:
                                response['quality'] = "Low"
        
        return jsonify(response)
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/reset', methods=['POST'])
def reset_session():
    data = request.json
    if not data or 'session_id' not in data:
        return jsonify({'error': 'session_id required'}), 400
    
    session_id = data['session_id']
    if session_id in buffer_storage:
        del buffer_storage[session_id]
    
    return jsonify({'status': 'success', 'message': 'Session reset'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.6.68.72:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1